# CGM Data Preprocessing

This notebook documents the preprocessing steps used to clean the raw CGM dataset before model training.

The workflow:
1. Loads the raw CGM data file.
2. Selects the relevant patient ID, time, and glucose columns.
3. Renames columns for consistency with the modeling pipeline.
4. Converts day and time information into a numeric timestamp.
5. Sorts glucose measurements by patient and time.
6. Removes implausible glucose values above 400 mg/dL.
7. Exports both the full cleaned dataset and a smaller 40-patient subset.

This notebook is included for transparency and reproducibility because preprocessing was performed separately from the main model training scripts.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the raw CGM data.
# The file is pipe-delimited, so sep="|" is needed instead of the default comma separator.
df = pd.read_csv("HDeviceCGM.csv", sep="|")

df.head()

In [ ]:
# Select only the columns needed for the modeling pipeline:
# patient ID, days from enrollment, device time, and glucose value.
df = df[
    [
        "PtID",
        "DeviceDtTmDaysFromEnroll",
        "DeviceTm",
        "GlucoseValue",
    ]
]

df.head()

In [ ]:
# Rename columns to simpler names that are easier to use throughout the analysis.
df = df.rename(
    columns={
        "PtID": "patient_id",
        "DeviceDtTmDaysFromEnroll": "day",
        "DeviceTm": "time",
        "GlucoseValue": "glucose",
    }
)

df.head()

In [ ]:
# Convert the time column into seconds from midnight.
# This allows each CGM measurement to be represented using one continuous numeric timestamp.
df["time_seconds"] = pd.to_timedelta(df["time"]).dt.total_seconds()

df.head()

In [ ]:
# Remove rows with missing key values before creating the final timestamp.
# These columns are required for downstream modeling.
df = df.dropna(
    subset=[
        "patient_id",
        "day",
        "time_seconds",
        "glucose",
    ]
)

df.head()

In [ ]:
# Make sure day is numeric, then create a continuous timestamp in seconds.
# Each day contributes 86,400 seconds, and time_seconds adds the time within that day.
df["day"] = df["day"].astype(int)

df["timestamp_numeric"] = df["day"] * 86400 + df["time_seconds"]

df.head()

In [ ]:
# Keep only the final columns needed for model training.
df = df[
    [
        "patient_id",
        "timestamp_numeric",
        "glucose",
    ]
]

df.head()

In [ ]:
# Sort by patient and time so each patient's CGM readings are in chronological order.
df = df.sort_values(
    [
        "patient_id",
        "timestamp_numeric",
    ]
)

df.head()

In [ ]:
# Check for missing values before final filtering.
df.isna().sum()

In [ ]:
# Inspect the glucose distribution before removing implausible high values.
df["glucose"].describe()

In [ ]:
# Remove implausibly high CGM values.
# Values above 400 mg/dL were excluded to reduce the influence of likely sensor/data errors.
df = df[df["glucose"] <= 400]

df["glucose"].describe()

In [ ]:
# Save the full cleaned dataset.
df.to_csv("C:/Users/Sarah/Downloads/clean_cgm_data_full.csv", index=False)

In [ ]:
# Create a smaller reproducible subset of 40 randomly selected patients.
# The random seed ensures the same patients are selected every time the notebook is run.
np.random.seed(42)

patients = df["patient_id"].unique()

selected_patients = np.random.choice(
    patients,
    size=40,
    replace=False,
)

df_small = df[df["patient_id"].isin(selected_patients)]

df_small.head()

In [ ]:
# Save the smaller 40-patient dataset.
df_small.to_csv("C:/Users/Sarah/Downloads/clean_cgm_data_small.csv", index=False)

In [ ]:
# Validation checks
# These checks confirm that the cleaned datasets are formatted correctly for modeling.

assert list(df.columns) == [
    "patient_id",
    "timestamp_numeric",
    "glucose",
], "Full dataset columns are not formatted correctly."

assert df["patient_id"].isna().sum() == 0, "Missing patient IDs found."
assert df["timestamp_numeric"].isna().sum() == 0, "Missing timestamps found."
assert df["glucose"].isna().sum() == 0, "Missing glucose values found."
assert df["glucose"].max() <= 400, "Glucose values above 400 were not removed."

assert df_small["patient_id"].nunique() == 40, "Small dataset does not contain 40 unique patients."
assert set(df_small.columns) == set(df.columns), "Small dataset columns do not match full dataset columns."

print("Preprocessing validation passed.")
print(f"Full cleaned dataset shape: {df.shape}")
print(f"Small 40-patient dataset shape: {df_small.shape}")
print(f"Number of unique patients in full dataset: {df['patient_id'].nunique()}")
print(f"Number of unique patients in small dataset: {df_small['patient_id'].nunique()}")